# Uncertainty intervals for the five-league evaluation

This notebook measures how uncertain the model comparisons are. It resamples complete match weeks within every league and season, keeping the two models paired on identical matches.

The primary test is the pooled recalibrated market versus the pooled market-plus-player model. Equal-league results are primary; match-weighted results are secondary. Positive improvement means the enhanced model is better. The notebook uses development seasons only and does not access 2025/26.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'scotland_research').is_dir()
)
RESEARCH_DIR = PROJECT_ROOT / 'scotland_research'
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

from constants import DEFAULT_EVALUATION_DIR
from calibration import (
    calibration_decision,
    calibration_effect_table,
    chronological_temperature_scale,
)
from evaluation.report import write_csv_atomic
from selected_features import load_selected_features, validate_selected_features
from uncertainty import (
    prepare_paired_comparison,
    resample_league_season_weeks,
    summarize_individual_league_intervals,
    summarize_interval,
)

EVALUATION_DIR = DEFAULT_EVALUATION_DIR
OUTPUT_DIR = RESEARCH_DIR / 'visuals' / 'uncertainty_measurements'
TABLES_DIR = OUTPUT_DIR / 'tables'
FIGURES_DIR = OUTPUT_DIR / 'figures'
REPETITIONS = 10_000
SEED = 42

print(f'Evaluation input: {EVALUATION_DIR}')
print(f'Outputs: {OUTPUT_DIR}')

Evaluation input: C:\Users\skous\Super-League-odds-research\artifacts\five_league_development_evaluation
Outputs: C:\Users\skous\Super-League-odds-research\scotland_research\visuals\uncertainty_measurements


## Comparisons

The first comparison directly answers the research question. The remaining comparisons separate market recalibration, league-specific player effects and Dixon–Coles player effects.

In [2]:
COMPARISONS = [
    {
        'comparison': 'Pooled players versus recalibrated market',
        'priority': 'primary',
        'training_scope': 'pooled',
        'baseline_model': 'recalibrated_market',
        'enhanced_model': 'market_plus_player_form',
    },
    {
        'comparison': 'Pooled recalibration versus raw market',
        'priority': 'secondary',
        'training_scope': 'pooled',
        'baseline_model': 'closing_market',
        'enhanced_model': 'recalibrated_market',
    },
    {
        'comparison': 'Separate players versus recalibrated market',
        'priority': 'secondary',
        'training_scope': 'league_specific',
        'baseline_model': 'recalibrated_market',
        'enhanced_model': 'market_plus_player_form',
    },
    {
        'comparison': 'Dixon-Coles with players versus plain Dixon-Coles',
        'priority': 'secondary',
        'training_scope': 'league_specific',
        'baseline_model': 'dixon_coles',
        'enhanced_model': 'dixon_coles_player_form',
    },
]
pd.DataFrame(COMPARISONS)

,comparison,priority,training_scope,baseline_model,enhanced_model
0,Pooled players versus recalibrated market,primary,pooled,recalibrated_market,market_plus_player_form
1,Pooled recalibration versus raw market,secondary,pooled,closing_market,recalibrated_market
2,Separate players versus recalibrated market,secondary,league_specific,recalibrated_market,market_plus_player_form
3,Dixon-Coles with players versus plain Dixon-Coles,secondary,league_specific,dixon_coles,dixon_coles_player_form


## Load and verify the current evaluation

The selected-feature copy saved beside the predictions must exactly match the active fixed specification. If the feature list changed after the evaluation, this cell stops and asks for a rerun instead of analysing stale predictions.

In [3]:
predictions_path = EVALUATION_DIR / 'predictions.csv'
evaluated_features_path = EVALUATION_DIR / 'selected_features.csv'
missing_files = [
    str(path)
    for path in (predictions_path, evaluated_features_path)
    if not path.exists()
]
if missing_files:
    raise FileNotFoundError(
        'Run scotland_research/evaluate_models.py first. Missing: '
        + ', '.join(missing_files)
    )

current_features = load_selected_features()
try:
    _, evaluated_hash = validate_selected_features(
        pd.read_csv(evaluated_features_path, dtype='string')
    )
except ValueError as error:
    raise RuntimeError(
        'The saved evaluation does not use the current selected features. '
        'Rerun scotland_research/evaluate_models.py before measuring uncertainty.'
    ) from error
if evaluated_hash != current_features.semantic_sha256:
    raise RuntimeError(
        'The saved evaluation feature checksum is stale. '
        'Rerun scotland_research/evaluate_models.py.'
    )

predictions = pd.read_csv(predictions_path)
available = predictions.groupby(['training_scope', 'model']).size().rename('matches')
print(f'Loaded {len(predictions):,} prediction rows.')
print(f'Feature checksum: {evaluated_hash}')
available

Loaded 31,160 prediction rows.
Feature checksum: 2f3f3a757a26bcd3df93eb798ae8dd8135d82bb40ab4c695a39f0846c4655df4


training_scope   model                  
league_specific  closing_market             3895
                 dixon_coles                3895
                 dixon_coles_player_form    3895
                 market_plus_player_form    3895
                 recalibrated_market        3895
pooled           closing_market             3895
                 market_plus_player_form    3895
                 recalibrated_market        3895
Name: matches, dtype: int64

## Paired match-week bootstrap

For each comparison, the notebook calculates each match's log loss, Brier score and normalized RPS under both models. It then resamples match-week blocks with replacement inside every league-season. Both models always receive the same sampled matches.

In [4]:
summary_frames = []
sample_frames = []
match_audit_rows = []

for comparison_number, comparison in enumerate(COMPARISONS):
    paired = prepare_paired_comparison(
        predictions,
        training_scope=comparison['training_scope'],
        baseline_model=comparison['baseline_model'],
        enhanced_model=comparison['enhanced_model'],
    )
    comparison_seed = SEED + comparison_number
    samples = resample_league_season_weeks(
        paired,
        repetitions=REPETITIONS,
        seed=comparison_seed,
    )
    summary_frames.append(
        summarize_interval(
            paired,
            samples,
            comparison,
            repetitions=REPETITIONS,
            seed=comparison_seed,
        )
    )
    samples.insert(0, 'comparison', comparison['comparison'])
    sample_frames.append(samples)
    match_audit_rows.append(
        {
            **comparison,
            'matches': len(paired),
            'leagues': paired['league'].nunique(),
            'seasons': paired['season'].nunique(),
            'league_season_weeks': paired[
                ['league', 'season', 'match_week']
            ].drop_duplicates().shape[0],
        }
    )
    print(f"Finished: {comparison['comparison']}")

summary = pd.concat(summary_frames, ignore_index=True)
bootstrap_samples = pd.concat(sample_frames, ignore_index=True)
match_audit = pd.DataFrame(match_audit_rows)

Finished: Pooled players versus recalibrated market
Finished: Pooled recalibration versus raw market
Finished: Separate players versus recalibrated market
Finished: Dixon-Coles with players versus plain Dixon-Coles


## Primary result

An interval entirely above zero supports the enhanced model. An interval containing zero means the development data do not clearly distinguish the models.

In [5]:
primary = summary[
    summary['priority'].eq('primary')
    & summary['weighting'].eq('equal_league')
][
    [
        'metric',
        'matches',
        'observed_absolute_improvement',
        'lower_95_absolute',
        'upper_95_absolute',
        'observed_relative_improvement_pct',
        'lower_95_relative_pct',
        'upper_95_relative_pct',
        'samples_favouring_enhanced_pct',
        'interval_excludes_zero',
    ]
].copy()
primary

,metric,matches,observed_absolute_improvement,lower_95_absolute,upper_95_absolute,observed_relative_improvement_pct,lower_95_relative_pct,upper_95_relative_pct,samples_favouring_enhanced_pct,interval_excludes_zero
0,log_loss,3895,0.000911,-0.001459,0.003247,0.098545,-0.157370,0.351136,78.37,False
1,brier_score,3895,0.000505,-0.001019,0.001982,0.092767,-0.188126,0.364627,74.82,False
2,rps,3895,0.000481,-0.000188,0.001142,0.260292,-0.101326,0.616334,92.49,False


## All comparisons

In [6]:
summary.sort_values(
    ['priority', 'comparison', 'weighting', 'metric'],
    kind='stable',
)[
    [
        'comparison',
        'weighting',
        'metric',
        'observed_relative_improvement_pct',
        'lower_95_relative_pct',
        'upper_95_relative_pct',
        'samples_favouring_enhanced_pct',
        'interval_excludes_zero',
    ]
]

,comparison,weighting,metric,observed_relative_improvement_pct,lower_95_relative_pct,upper_95_relative_pct,samples_favouring_enhanced_pct,interval_excludes_zero
1,Pooled players versus recalibrated market,equal_league,brier_score,0.092767,-0.188126,0.364627,74.82,False
0,Pooled players versus recalibrated market,equal_league,log_loss,0.098545,-0.157370,0.351136,78.37,False
2,Pooled players versus recalibrated market,equal_league,rps,0.260292,-0.101326,0.616334,92.49,False
4,Pooled players versus recalibrated market,match_weighted,brier_score,0.042113,-0.246060,0.320141,61.01,False
3,Pooled players versus recalibrated market,match_weighted,log_loss,0.049286,-0.207965,0.301689,64.79,False
5,Pooled players versus recalibrated market,match_weighted,rps,0.197683,-0.167131,0.554625,86.37,False
19,Dixon-Coles with players versus plain Dixon-Coles,equal_league,brier_score,0.178110,-0.272184,0.618541,77.79,False
18,Dixon-Coles with players versus plain Dixon-Coles,equal_league,log_loss,0.020731,-0.420558,0.439930,53.37,False
20,Dixon-Coles with players versus plain Dixon-Coles,equal_league,rps,0.218287,-0.378146,0.808514,75.27,False
22,Dixon-Coles with players versus plain Dixon-Coles,match_weighted,brier_score,0.239999,-0.231342,0.695651,83.90,False


## Chronological post-hoc calibration

A single temperature value can soften or sharpen probabilities without changing the most likely result. For 2023/24 it is fitted only on 2022/23 out-of-sample predictions. For 2024/25 it is fitted on 2022/23–2023/24 predictions. Both pooled primary models receive exactly the same procedure.

In [7]:
CALIBRATION_MODELS = ['recalibrated_market', 'market_plus_player_form']
calibrated_prediction_frames = []
temperature_frames = []
calibration_fold_frames = []
calibration_league_frames = []
calibration_pair_data = {}
decision_rows = []

for model_number, model in enumerate(CALIBRATION_MODELS):
    result = chronological_temperature_scale(
        predictions,
        training_scope='pooled',
        model=model,
    )
    calibrated_prediction_frames.append(result.predictions)
    temperature_frames.append(result.fitted_temperatures)
    calibrated_seasons = sorted(result.predictions['season'].unique())
    original = predictions[
        predictions['training_scope'].eq('pooled')
        & predictions['model'].eq(model)
        & predictions['season'].isin(calibrated_seasons)
    ].copy()
    combined = pd.concat([original, result.predictions], ignore_index=True)
    paired_calibration = prepare_paired_comparison(
        combined,
        training_scope='pooled',
        baseline_model=model,
        enhanced_model=f'{model}_temperature_scaled',
    )
    calibration_pair_data[model] = paired_calibration
    decision = calibration_decision(paired_calibration)
    decision_rows.append({'model': model, **decision})
    fold_table = calibration_effect_table(
        paired_calibration, group_column='season', equal_league=True
    )
    fold_table.insert(0, 'model', model)
    calibration_fold_frames.append(fold_table)
    league_table = calibration_effect_table(
        paired_calibration, group_column='league', equal_league=False
    )
    league_table.insert(0, 'model', model)
    calibration_league_frames.append(league_table)

calibrated_predictions = pd.concat(calibrated_prediction_frames, ignore_index=True)
fitted_temperatures = pd.concat(temperature_frames, ignore_index=True)
calibration_by_fold = pd.concat(calibration_fold_frames, ignore_index=True)
calibration_by_league = pd.concat(calibration_league_frames, ignore_index=True)
calibration_decisions = pd.DataFrame(decision_rows)
calibration_decisions

,model,retain_calibration,equal_league_log_loss_improvement,equal_league_brier_improvement,worst_fold_log_loss_improvement,max_allowed_fold_deterioration,leagues_improved,leagues_required,equal_league_log_loss_improved,equal_league_brier_not_worse,no_fold_log_loss_deterioration_over_limit,log_loss_improved_in_league_majority
0,recalibrated_market,False,-0.001579,-0.000675,-0.001447,0.001,1,3,False,False,False,False
1,market_plus_player_form,False,-0.002100,-0.000914,-0.001955,0.001,1,3,False,False,False,False


### Calibrated primary comparison

This repeats the research comparison using only 2023/24 and 2024/25, because 2022/23 has no earlier out-of-sample season available for fitting the calibrator.

In [8]:
calibrated_primary_comparison = {
    'comparison': 'Calibrated pooled players versus calibrated market',
    'priority': 'calibration',
    'training_scope': 'pooled',
    'baseline_model': 'recalibrated_market_temperature_scaled',
    'enhanced_model': 'market_plus_player_form_temperature_scaled',
}
calibrated_primary_data = prepare_paired_comparison(
    calibrated_predictions,
    training_scope='pooled',
    baseline_model=calibrated_primary_comparison['baseline_model'],
    enhanced_model=calibrated_primary_comparison['enhanced_model'],
)
calibrated_primary_samples = resample_league_season_weeks(
    calibrated_primary_data, repetitions=REPETITIONS, seed=SEED + 20
)
calibrated_primary_summary = summarize_interval(
    calibrated_primary_data,
    calibrated_primary_samples,
    calibrated_primary_comparison,
    repetitions=REPETITIONS,
    seed=SEED + 20,
)
calibrated_primary_summary[
    calibrated_primary_summary['weighting'].eq('equal_league')
][
    [
        'metric', 'observed_relative_improvement_pct',
        'lower_95_relative_pct', 'upper_95_relative_pct',
        'samples_favouring_enhanced_pct', 'interval_excludes_zero',
    ]
]

,metric,observed_relative_improvement_pct,lower_95_relative_pct,upper_95_relative_pct,samples_favouring_enhanced_pct,interval_excludes_zero
0,log_loss,-0.004317,-0.372996,0.360758,48.56,False
1,brier_score,-0.003796,-0.398164,0.378911,48.87,False
2,rps,0.152676,-0.369998,0.669872,71.95,False


## Individual-league intervals

These are supporting results. Each country resamples its own season-week blocks, so intervals will naturally be wider than the combined analysis.

In [9]:
primary_comparison = COMPARISONS[0]
primary_pair_data = prepare_paired_comparison(
    predictions,
    training_scope=primary_comparison['training_scope'],
    baseline_model=primary_comparison['baseline_model'],
    enhanced_model=primary_comparison['enhanced_model'],
)
uncalibrated_by_league = summarize_individual_league_intervals(
    primary_pair_data,
    primary_comparison,
    repetitions=REPETITIONS,
    seed=SEED + 30,
)
uncalibrated_by_league.insert(0, 'probability_version', 'uncalibrated')
calibrated_by_league = summarize_individual_league_intervals(
    calibrated_primary_data,
    calibrated_primary_comparison,
    repetitions=REPETITIONS,
    seed=SEED + 40,
)
calibrated_by_league.insert(0, 'probability_version', 'temperature_scaled')
uncertainty_by_league = pd.concat(
    [uncalibrated_by_league, calibrated_by_league], ignore_index=True
)
uncertainty_by_league = uncertainty_by_league[
    uncertainty_by_league['metric'].isin(['log_loss', 'brier_score'])
]
uncertainty_by_league[
    [
        'probability_version', 'league', 'metric',
        'observed_relative_improvement_pct',
        'lower_95_relative_pct', 'upper_95_relative_pct',
        'interval_excludes_zero',
    ]
]

,probability_version,league,metric,observed_relative_improvement_pct,lower_95_relative_pct,upper_95_relative_pct,interval_excludes_zero
0,uncalibrated,belgium,log_loss,0.181269,-0.348024,0.706036,False
1,uncalibrated,belgium,brier_score,0.058400,-0.562821,0.656771,False
3,uncalibrated,netherlands,log_loss,-0.440891,-0.980443,0.112221,False
4,uncalibrated,netherlands,brier_score,-0.399196,-0.979725,0.197251,False
6,uncalibrated,portugal,log_loss,0.755937,0.136275,1.361973,True
7,uncalibrated,portugal,brier_score,0.734925,0.095563,1.355886,True
9,uncalibrated,scotland,log_loss,-0.081306,-0.640202,0.461751,False
10,uncalibrated,scotland,brier_score,-0.048827,-0.699040,0.567239,False
12,uncalibrated,turkey,log_loss,0.088320,-0.462556,0.588266,False
13,uncalibrated,turkey,brier_score,0.145059,-0.436324,0.695645,False


## Interval figure

The dots are observed equal-league improvements. Horizontal lines show the middle 95% of the paired bootstrap results.

In [ ]:
plot_summary = pd.concat([summary, calibrated_primary_summary], ignore_index=True)
plot_data = plot_summary[plot_summary['weighting'].eq('equal_league')].copy()
comparison_order = [
    *[item['comparison'] for item in COMPARISONS],
    calibrated_primary_comparison['comparison'],
]
metric_labels = {
    'log_loss': 'Log loss',
    'brier_score': 'Brier score',
    'rps': 'RPS',
}
fig, axes = plt.subplots(1, 3, figsize=(17, 6), sharey=True, constrained_layout=True)
for axis, (metric, metric_label) in zip(axes, metric_labels.items()):
    current = (
        plot_data[plot_data['metric'].eq(metric)]
        .set_index('comparison')
        .loc[comparison_order]
        .reset_index()
    )
    y = np.arange(len(current))
    observed = current['observed_absolute_improvement'].to_numpy()
    lower = current['lower_95_absolute'].to_numpy()
    upper = current['upper_95_absolute'].to_numpy()
    point_colors = ['royalblue' if priority == 'primary' else 'dimgray' for priority in current['priority']]
    axis.errorbar(
        observed,
        y,
        xerr=np.vstack([observed - lower, upper - observed]),
        fmt='none',
        ecolor='gray',
        elinewidth=2,
        capsize=4,
    )
    axis.scatter(observed, y, color=point_colors, s=45, zorder=3)
    axis.axvline(0, color='black', linestyle='--', linewidth=1)
    axis.set_title(metric_label)
    axis.set_xlabel('Improvement over baseline')
    axis.spines[['top', 'right']].set_visible(False)
    axis.set_yticks(y, comparison_order)
    axis.invert_yaxis()
fig.suptitle('Paired match-week uncertainty intervals — equal-league results')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure_path = FIGURES_DIR / 'uncertainty_intervals.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {figure_path}')

## Save auditable outputs

In [ ]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)
write_csv_atomic(summary, TABLES_DIR / 'uncertainty_intervals.csv')
write_csv_atomic(match_audit, TABLES_DIR / 'comparison_match_audit.csv')
write_csv_atomic(calibration_by_fold, TABLES_DIR / 'calibration_by_fold.csv')
write_csv_atomic(calibration_by_league, TABLES_DIR / 'calibration_by_league.csv')
write_csv_atomic(calibrated_predictions, TABLES_DIR / 'calibrated_predictions.csv')
write_csv_atomic(calibration_decisions, TABLES_DIR / 'calibration_decisions.csv')
write_csv_atomic(fitted_temperatures, TABLES_DIR / 'fitted_temperatures.csv')
write_csv_atomic(calibrated_primary_summary, TABLES_DIR / 'calibrated_primary_intervals.csv')
write_csv_atomic(uncertainty_by_league, TABLES_DIR / 'uncertainty_by_league.csv')

settings = {
    'development_only': True,
    'prediction_source': str(predictions_path),
    'selected_features_sha256': evaluated_hash,
    'repetitions': REPETITIONS,
    'base_seed': SEED,
    'resampling_unit': 'match week within league and season',
    'primary_weighting': 'equal league',
    'secondary_weighting': 'match weighted',
    'primary_metric': 'log loss',
    'secondary_metrics': ['Brier score', 'normalized RPS'],
    'comparisons': COMPARISONS,
    'calibration_method': 'single-temperature scaling',
    'calibration_training': 'earlier out-of-sample prediction seasons only',
    'calibration_scope': 'pooled recalibrated market and pooled player model',
    'calibration_decision_rule': {
        'equal_league_log_loss_improved': True,
        'equal_league_brier_not_worse': True,
        'maximum_fold_log_loss_deterioration': 0.001,
        'log_loss_improved_in_at_least_three_of_five_leagues': True,
    },
}
settings_path = TABLES_DIR / 'run_settings.json'
settings_path.write_text(json.dumps(settings, indent=2), encoding='utf-8')
calibration_settings_path = TABLES_DIR / 'calibration_settings.json'
calibration_settings_path.write_text(
    json.dumps(
        {
            'method': settings['calibration_method'],
            'training': settings['calibration_training'],
            'scope': settings['calibration_scope'],
            'decision_rule': settings['calibration_decision_rule'],
            'decisions': calibration_decisions.to_dict('records'),
        },
        indent=2,
    ),
    encoding='utf-8',
)
print(f'Saved interval tables and settings to {TABLES_DIR}')

## Interpretation rules

- Focus first on the equal-league log-loss interval for **Pooled players versus recalibrated market**.
- If its 95% interval includes zero, describe the player improvement as uncertain rather than meaningful.
- Brier score and RPS should point in the same general direction; they are supporting measures, not reasons to override log loss.
- Match-weighted results show whether fixture-heavy leagues change the conclusion.
- Retain temperature scaling only when every recorded calibration rule passes; otherwise keep the original probabilities.
- Country intervals are supporting evidence and should not be used for further feature selection.
- Secondary comparisons explain the model behaviour but do not replace the primary research test.
- These intervals describe development-sample uncertainty. The untouched 2025/26 evaluation remains the final independent test.